# Mineral Spectral QA — Qwen2.5-7B QLoRA on T4

CRISM 스펙트럼 → 텍스트 인코딩 → Qwen2.5-7B QLoRA 학습 → 광물 분류 + 설명

**Runtime**: GPU → T4 (16GB VRAM)

**예상 소요**: ~6시간 (10K samples, 10 epochs)

## 0. Setup

In [ ]:
!pip install -q torch transformers peft accelerate bitsandbytes scipy scikit-learn tqdm matplotlib

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
!git clone https://github.com/jejuchild/mineral-vqa-classification.git
%cd mineral-vqa-classification

In [ ]:
# Download training data (247MB)
import os, urllib.request
NPZ_URL = "https://github.com/jejuchild/crism-mineral-data/releases/download/v1.0/crism_training_data_f16.npz"
NPZ_PATH = "crism_training_data_f16.npz"
if not os.path.exists(NPZ_PATH):
    print("Downloading training data...")
    urllib.request.urlretrieve(NPZ_URL, NPZ_PATH)
    print(f"Downloaded: {os.path.getsize(NPZ_PATH) / 1e6:.0f} MB")
else:
    print(f"Already exists: {os.path.getsize(NPZ_PATH) / 1e6:.0f} MB")

## 1. Generate QA Pairs

In [ ]:
import sys
sys.path.insert(0, "src")

import json
import numpy as np
import random
from tqdm.auto import tqdm

from config import BANDS, CLASS_NAME, SEED, MAX_SAMPLES
from spectrum_encoder import encode_spectrum, preprocess_batch, detect_absorptions, match_cause

In [ ]:
# Load data
data = np.load(NPZ_PATH, allow_pickle=True)
X_raw = data["X"].astype(np.float32)
y = data["y"]
obs_idx = data["obs_idx"]

print(f"Total pixels: {len(X_raw)}")
print(f"Classes: {np.unique(y)}")
print(f"Observations: {len(np.unique(obs_idx))}")

In [ ]:
# Stratified subsample
MAX_N = 10000  # adjust: 5000 for ~3h, 10000 for ~6h
rng = np.random.RandomState(SEED)

n_total = len(X_raw)
indices = []
for cls in np.unique(y):
    cls_idx = np.where(y == cls)[0]
    n_cls = max(1, int(MAX_N * len(cls_idx) / n_total))
    chosen = rng.choice(cls_idx, size=min(n_cls, len(cls_idx)), replace=False)
    indices.append(chosen)
indices = np.concatenate(indices)
rng.shuffle(indices)
indices = indices[:MAX_N]

print(f"Selected: {len(indices)} spectra")

In [ ]:
# Preprocess (fill interp + CR)
print("Preprocessing...")
X_sel = preprocess_batch(X_raw[indices])
y_sel = y[indices]
obs_sel = obs_idx[indices]
print("Done.")

In [ ]:
# Load knowledge bases
with open("knowledge/mineral_kb.json") as f:
    kb = json.load(f)
with open("knowledge/absorption_bands.json") as f:
    absorption_catalog = json.load(f)

print(f"KB minerals: {len(kb)}")

In [ ]:
# Generate QA pairs
all_qa = []
skipped = 0

for i in tqdm(range(len(X_sel)), desc="Generating QA"):
    label = int(y_sel[i])
    mineral = CLASS_NAME.get(label)
    if mineral is None:
        skipped += 1
        continue
    kb_entry = kb.get(mineral)
    if kb_entry is None:
        skipped += 1
        continue

    spectrum_cr = X_sel[i]
    spectrum_text = encode_spectrum(spectrum_cr)
    detected = detect_absorptions(spectrum_cr)
    group = kb_entry.get("group", "Unknown")
    subgroup = kb_entry.get("subgroup", "Unknown")

    # Type A: Classification
    all_qa.append({"spectrum": spectrum_text, "question": "What mineral is this?", "answer": mineral, "type": "A", "obs_idx": int(obs_sel[i])})
    all_qa.append({"spectrum": spectrum_text, "question": "What mineral group does this belong to?", "answer": f"{group}, {subgroup} subgroup", "type": "A", "obs_idx": int(obs_sel[i])})
    formula = kb_entry.get("formula")
    if formula:
        all_qa.append({"spectrum": spectrum_text, "question": "What is the chemical formula of this mineral?", "answer": formula, "type": "A", "obs_idx": int(obs_sel[i])})
    mars_ctx = kb_entry.get("mars_context")
    if mars_ctx:
        all_qa.append({"spectrum": spectrum_text, "question": "Where on Mars is this mineral typically found?", "answer": mars_ctx, "type": "A", "obs_idx": int(obs_sel[i])})

    # Type B: Absorption features
    if detected:
        feat_strs = []
        for ab in detected[:5]:
            wl, depth = ab["wavelength"], ab["depth"]
            cause = match_cause(wl)
            feat_strs.append(f"{wl:.2f}um (depth={depth:.3f}{', ' + cause if cause else ''})")
        all_qa.append({"spectrum": spectrum_text, "question": "What absorption features are present in this spectrum?", "answer": f"Absorption features detected: {'; '.join(feat_strs)}.", "type": "B", "obs_idx": int(obs_sel[i])})
    desc = kb_entry.get("spectral_description")
    if desc:
        all_qa.append({"spectrum": spectrum_text, "question": "Describe the spectral characteristics of this mineral.", "answer": desc, "type": "B", "obs_idx": int(obs_sel[i])})
    band_assigns = kb_entry.get("band_assignments", {})
    diag_bands = kb_entry.get("diagnostic_bands_um", [])
    if band_assigns and diag_bands:
        band_str = ", ".join(f"{b}um" for b in diag_bands)
        assign_strs = [f"{k}um: {v}" for k, v in band_assigns.items()]
        all_qa.append({"spectrum": spectrum_text, "question": f"What are the diagnostic absorption bands for {mineral}?", "answer": f"Diagnostic bands for {mineral}: {band_str}. Assignments: {'; '.join(assign_strs)}.", "type": "B", "obs_idx": int(obs_sel[i])})

    # Type C: Differential diagnosis
    for other, expl in kb_entry.get("distinguish_from", {}).items():
        all_qa.append({"spectrum": spectrum_text, "question": f"How do you distinguish this from {other}?", "answer": expl, "type": "C", "obs_idx": int(obs_sel[i])})

    # Type D: CRISM parameters
    params = kb_entry.get("crism_params", [])
    if params:
        param_details = absorption_catalog.get("spectral_parameters_viviano_beck", {})
        p_strs = [f"{p} ({param_details.get(p, {}).get('detects', 'detection')})" if p in param_details else p for p in params]
        all_qa.append({"spectrum": spectrum_text, "question": "What CRISM spectral parameters would detect this mineral?", "answer": f"CRISM spectral parameters for {mineral}: {'; '.join(p_strs)}.", "type": "D", "obs_idx": int(obs_sel[i])})

print(f"\nTotal QA pairs: {len(all_qa)}")
print(f"Skipped: {skipped}")
from collections import Counter
print(f"Type distribution: {dict(Counter(q['type'] for q in all_qa))}")

## 2. Observation-wise Split

In [ ]:
# Split by observation (no scene leakage)
random.seed(SEED)

obs_to_qa = {}
for qa in all_qa:
    obs = qa["obs_idx"]
    obs_to_qa.setdefault(obs, []).append(qa)

obs_list = sorted(obs_to_qa.keys())
random.shuffle(obs_list)

n_train = int(len(obs_list) * 0.8)
n_val = int(len(obs_list) * 0.1)

train_obs = set(obs_list[:n_train])
val_obs = set(obs_list[n_train:n_train + n_val])
test_obs = set(obs_list[n_train + n_val:])

train_qa = [qa for obs in train_obs for qa in obs_to_qa[obs]]
val_qa = [qa for obs in val_obs for qa in obs_to_qa[obs]]
test_qa = [qa for obs in test_obs for qa in obs_to_qa[obs]]

random.shuffle(train_qa)
random.shuffle(val_qa)

print(f"Train: {len(train_qa)}, Val: {len(val_qa)}, Test: {len(test_qa)}")
print(f"Obs split: {len(train_obs)} / {len(val_obs)} / {len(test_obs)}")

## 3. Load Qwen2.5-7B with QLoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading model (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

# LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# VRAM check
print(f"\nGPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print(f"GPU memory reserved:   {torch.cuda.memory_reserved() / 1e9:.1f} GB")

## 4. Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

SYSTEM_PROMPT = (
    "You are a Mars mineral spectroscopy expert. "
    "Given a CRISM spectrum (continuum-removed, 1.02-3.92um), "
    "answer questions about mineral identification, absorption features, "
    "and spectral characteristics."
)
MAX_LEN = 768

def format_train(spec, q, a):
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{spec}\n\n{q}<|im_end|>\n"
        f"<|im_start|>assistant\n{a}<|im_end|>"
    )

def format_prompt(spec, q):
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{spec}\n\n{q}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

class MineralQADataset(Dataset):
    def __init__(self, qa_pairs, tokenizer, max_len=MAX_LEN):
        self.qa = qa_pairs
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.qa)

    def __getitem__(self, idx):
        item = self.qa[idx]
        full = format_train(item["spectrum"], item["question"], item["answer"])
        enc = self.tok(full, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")

        input_ids = enc["input_ids"].squeeze(0)
        attn_mask = enc["attention_mask"].squeeze(0)
        labels = input_ids.clone()

        # Mask prompt (only train on answer)
        prompt = format_prompt(item["spectrum"], item["question"])
        prompt_len = len(self.tok(prompt, truncation=True, max_length=self.max_len)["input_ids"])
        labels[:prompt_len] = -100
        labels[attn_mask == 0] = -100

        return {"input_ids": input_ids, "attention_mask": attn_mask, "labels": labels}

train_ds = MineralQADataset(train_qa, tokenizer)
val_ds = MineralQADataset(val_qa, tokenizer)

BATCH_SIZE = 2
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 5. Training

In [ ]:
from transformers import get_linear_schedule_with_warmup

EPOCHS = 10
GRAD_ACCUM = 16
LR = 1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS // GRAD_ACCUM
warmup_steps = int(total_steps * 0.03)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, max(total_steps, 1))

print(f"Total optimization steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

In [ ]:
import time

best_val_loss = float("inf")
train_losses = []
val_losses = []
device = next(model.parameters()).device

for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for step, batch in enumerate(pbar):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        epoch_loss += outputs.loss.item()
        pbar.set_postfix({"loss": f"{outputs.loss.item():.4f}"})

    # Flush remaining grads
    if (step + 1) % GRAD_ACCUM != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    avg_train = epoch_loss / len(train_loader)
    train_losses.append(avg_train)

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Val", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            val_loss += outputs.loss.item()
    avg_val = val_loss / max(len(val_loader), 1)
    val_losses.append(avg_val)

    elapsed = time.time() - t0
    print(f"Epoch {epoch+1}: train={avg_train:.4f}, val={avg_val:.4f}, time={elapsed:.0f}s")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        model.save_pretrained("models/vqa_lora/best")
        tokenizer.save_pretrained("models/vqa_lora/best")
        print(f"  -> Saved best (val={best_val_loss:.4f})")

# Save final
model.save_pretrained("models/vqa_lora/final")
tokenizer.save_pretrained("models/vqa_lora/final")
print("\nTraining complete!")

In [ ]:
# Loss curves
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(train_losses)+1), train_losses, "o-", label="Train")
ax.plot(range(1, len(val_losses)+1), val_losses, "s-", label="Val")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title("Mineral QA Training Loss")
plt.tight_layout()
plt.show()

## 6. Evaluation on Test Set

In [ ]:
from peft import PeftModel

# Reload best checkpoint
del model
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, "models/vqa_lora/best")
model.eval()
device = next(model.parameters()).device
print("Best model loaded.")

In [ ]:
def ask(spectrum_text, question, max_new_tokens=150):
    prompt = format_prompt(spectrum_text, question)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        gen = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, num_beams=3, early_stopping=True,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    prompt_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(gen[0][prompt_len:], skip_special_tokens=True).strip()

In [ ]:
# Test: mineral classification accuracy (Type A only)
test_type_a = [q for q in test_qa if q["type"] == "A" and q["question"] == "What mineral is this?"]
print(f"Test Type A (classification): {len(test_type_a)} samples")

correct = 0
results = []

for item in tqdm(test_type_a[:100], desc="Evaluating"):  # sample 100 for speed
    pred = ask(item["spectrum"], item["question"], max_new_tokens=30)
    true = item["answer"]
    match = true.lower() in pred.lower()
    if match:
        correct += 1
    results.append({"true": true, "pred": pred, "match": match})

acc = correct / len(results)
print(f"\nClassification Accuracy: {acc:.1%} ({correct}/{len(results)})")

# Show some examples
print("\n--- Correct ---")
for r in [r for r in results if r["match"]][:3]:
    print(f"  True: {r['true']:20s} Pred: {r['pred']}")
print("\n--- Wrong ---")
for r in [r for r in results if not r["match"]][:3]:
    print(f"  True: {r['true']:20s} Pred: {r['pred']}")

In [ ]:
# Test: qualitative examples (all QA types)
sample_indices = random.sample(range(len(test_qa)), min(10, len(test_qa)))

for idx in sample_indices:
    item = test_qa[idx]
    pred = ask(item["spectrum"], item["question"])
    print(f"[Type {item['type']}] Q: {item['question']}")
    print(f"  True: {item['answer'][:120]}..." if len(item['answer']) > 120 else f"  True: {item['answer']}")
    print(f"  Pred: {pred[:120]}..." if len(pred) > 120 else f"  Pred: {pred}")
    print()

## 7. Interactive Inference

In [ ]:
# Pick a random spectrum from the dataset
test_idx = random.randint(0, len(X_raw) - 1)
test_label = CLASS_NAME.get(int(y[test_idx]), f"Class_{y[test_idx]}")
print(f"Test pixel index: {test_idx}, True label: {test_label}")

from spectrum_encoder import encode_spectrum_from_raw
test_spectrum = encode_spectrum_from_raw(X_raw[test_idx])

questions = [
    "What mineral is this?",
    "What absorption features are present in this spectrum?",
    "What mineral group does this belong to?",
    "Describe the spectral characteristics of this mineral.",
    "What CRISM spectral parameters would detect this mineral?",
]

for q in questions:
    a = ask(test_spectrum, q)
    print(f"\nQ: {q}")
    print(f"A: {a}")

## 8. Save to Google Drive (optional)

In [ ]:
# Uncomment to save to Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r models/vqa_lora/best /content/drive/MyDrive/mineral_vqa_best
# print("Saved to Google Drive!")